<a href="https://colab.research.google.com/github/rafaellopesdesa/hnsbi-toolkit/blob/main/examples/notebooks/neural_importance_sampling_asimov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural importance-sampling Asimov construction

The YAML supplies the design points, defensive mixture $\epsilon$, pilot size, proposal flow, and target quadrature size. This notebook trains every required model through the native package and reports validation and ESS.

> If a previous Colab run already loaded packages that the setup must replace, the setup cell restarts the kernel once. After Colab reconnects, run the setup cell again.

In [ ]:
from importlib.metadata import PackageNotFoundError, version as installed_version
from pathlib import Path
import os, signal, subprocess, sys

def distribution_version(name):
    try:
        return installed_version(name)
    except PackageNotFoundError:
        return None

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    loaded_versions = {
        name: getattr(sys.modules.get(name), '__version__', None)
        for name in ('numpy', 'jax', 'jaxlib')
    }
    jax_was_loaded = any(
        name == 'jax' or name.startswith(('jax.', 'jaxlib', 'jax_plugins'))
        for name in sys.modules
    )
    ROOT = Path('/content/drive/MyDrive/hsbi-toolkit')
    REPO = ROOT / 'hnsbi-toolkit'
    ROOT.mkdir(parents=True, exist_ok=True)
    if not (REPO / '.git').is_dir():
        subprocess.run(['git', 'clone', 'https://github.com/rafaellopesdesa/hnsbi-toolkit.git', str(REPO)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[lhc,flows]'], check=True)

    # Colab installs JAX's CUDA plugin separately from jaxlib. If an
    # earlier dependency resolution changed jaxlib, realign the plugin
    # before JAX discovers it; mixed PJRT versions fail at execution.
    jaxlib_version = distribution_version('jaxlib')
    plugin_extras = {
        'jax-cuda12-plugin': 'cuda12-local',
        'jax-cuda13-plugin': 'cuda13-local',
    }
    repaired_plugins = []
    for plugin, extra in plugin_extras.items():
        plugin_version = distribution_version(plugin)
        if plugin_version is not None and plugin_version != jaxlib_version:
            print(f'Aligning {plugin} {plugin_version} with jaxlib {jaxlib_version}.')
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                 f'jax[{extra}]=={jaxlib_version}'],
                check=True,
            )
            if distribution_version(plugin) != jaxlib_version:
                raise RuntimeError(f'Could not align {plugin} with jaxlib.')
            repaired_plugins.append(plugin)

    changed_loaded_packages = [
        name for name, loaded in loaded_versions.items()
        if loaded is not None and loaded != distribution_version(name)
    ]
    if changed_loaded_packages or (repaired_plugins and jax_was_loaded):
        reasons = changed_loaded_packages + repaired_plugins
        print(
            f'Updated {", ".join(dict.fromkeys(reasons))}. '
            'Restarting the Colab runtime once to load a consistent NumPy/JAX stack. '
            'After it reconnects, run this setup cell again.',
            flush=True,
        )
        os.kill(os.getpid(), signal.SIGKILL)
else:
    REPO = Path.cwd()
    if not (REPO / 'pyproject.toml').exists():
        REPO = Path.cwd().parents[1]
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'examples' / 'lhc_analysis'))


In [ ]:
from generate_distributions import generate
from hnsbi import Project

EXAMPLE = REPO / 'examples' / 'lhc_analysis'
generate(EXAMPLE / 'data', signal_events=12_000, background_events=30_000, reference_events=40_000)
project = Project.load(EXAMPLE / 'analysis.yaml')
reference_artifacts = project.train_reference()
reference = reference_artifacts.training.flow
ratios = project.train_ratios(reference, normalization_events=40_000, seed=20260729)
systematic_training = project.train_systematics()
runtime_systematics = project.build_runtime_systematics(systematic_training)


## Train and validate the proposal

The defensive density is

$$
g_\epsilon(x)=(1-\epsilon)g_\psi(x)+\epsilon q_\phi(x).
$$

All proposal diagnostics and ONNX parity results are included in the returned artifact bundle.

In [ ]:
nis = project.train_nis_asimov(reference=reference, ratios=ratios.evaluators, truth_point={'mu': 1.0, 'response': 0.0, 'resolution': 0.0, 'theory': 0.0}, asimov_point={'mu': 1.0, 'response': 0.0, 'resolution': 0.0, 'theory': 0.0}, systematics=runtime_systematics)
print('raw count:', nis.asimov.raw_count)
print('ESS:', nis.asimov.ess)
print('validation:', nis.validation_provenance)
print('workspace-ready arrays:', nis.asimov_array_paths)
